<a href="https://colab.research.google.com/github/Josephleme/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duka Electronics Sales Exploration


### Executive Summary

* Duka Electronics generated an aggregate total revenue of KES 72,077,600 during the first half of 2026.
* Operational volume reached exactly 1,000 completed orders across all sales channels.
* The Nairobi Central branch stood out as the primary location, managing 368 total orders.
* Financial and transactional performance tracking spanned a continuous timeline of 6 months.
* The business successfully managed consumer market demand across 5 unique product categories.


### First-Five-Minutes Inspection

* **Shape:** The dataset contains 1,000 rows and 6 columns.
* **Data Types:** Variables consist of text strings (order_id, branch, product, channel, order_date), integers (units), and floating-point numbers (unit_price).
* **Missing Values:** There are 0 missing or null values across all columns in this dataset.
* **Date Range:** The data covers transactions from January 1, 2026, through June 30, 2026.
* **Key Categories:** Operations span 5 distinct branches and 5 unique product types distributed across In-Store and Online channels.


In [ ]:
import pandas as pd
import numpy as np

def generate_datasets():
    rng_duka = np.random.default_rng(2026)
    n_duka = 1000

    duka_branches = ["Nairobi Central", "Mombasa CBD", "Kisumu Mega", "Nakuru West", "Kiambu Town"]
    duka_products = {
        "Premium Laptop": 85000, "Midrange Smartphone": 32000,
        "Budget Phone": 12000, "Wireless Earbuds": 6500, "Smart Watch": 15000
    }
    prod_names = list(duka_products.keys())

    duka_sales = pd.DataFrame({
        "order_id": [f"DK{i:04d}" for i in range(1, n_duka + 1)],
        "branch": rng_duka.choice(duka_branches, n_duka, p=[0.4, 0.2, 0.15, 0.15, 0.10]),
        "product": rng_duka.choice(prod_names, n_duka, p=[0.15, 0.30, 0.25, 0.15, 0.15]),
        "units": rng_duka.integers(1, 5, n_duka),
        "channel": rng_duka.choice(["In-Store", "Online"], n_duka, p=[0.6, 0.4]),
        "order_date": (pd.to_datetime("2026-01-01") +
                       pd.to_timedelta(rng_duka.integers(0, 181, n_duka), unit="D")).strftime("%Y-%m-%d")
    })

    product_base_prices = duka_sales["product"].map(duka_products)
    duka_sales["unit_price"] = (product_base_prices * rng_duka.uniform(0.9, 1.05, n_duka)).round(-2)
    duka_sales.to_csv("duka_sales.csv", index=False)
    print("✔ duka_sales.csv generated successfully.")

    rng = np.random.default_rng(44)
    n = 600

    agents = pd.DataFrame({
        "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
        "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
        "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
    })

    txns = pd.DataFrame({
        "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
        "agent_id": rng.choice(agents["agent_id"], n),
        "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n),
        "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
        "txn_date": (pd.to_datetime("2026-03-01") + pd.to_timedelta(rng.integers(0, 60, n), unit="D")).strftime("%Y-%m-%d"),
    })

    idx = rng.choice(n, 30, replace=False)
    txns.loc[idx[:15], "amount"] = np.nan
    txns.loc[idx[15:20], "amount"] = "N/A"
    txns.loc[idx[20:25], "amount"] = "250000"
    txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
    txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
    txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

    agents.to_csv("agents.csv", index=False)
    txns.to_csv("agent_transactions.csv", index=False)


if __name__ == "__main__":
    generate_datasets()

df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
df["revenue"] = df["units"] * df["unit_price"]

df["revenue"] = df["units"] * df["unit_price"]
df["month"] = df["order_date"].dt.strftime("%B")
df["weekday"] = df["order_date"].dt.strftime("%A")
df["is_weekend"] = df["order_date"].dt.weekday.isin([5, 6])
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

df.head()

# print(f"Total Revenue: KES {df['revenue'].sum():,}")
# print(f"Total Orders: {len(df)}")
# print(f"Nairobi Orders: {df[df['branch'] == 'Nairobi Central'].shape[0]}")
# print(f"Months Covered: {df['order_date'].dt.to_period('M').nunique()}")
# print(f"Unique Products: {df['product'].nunique()}")


,order_id,branch,product,units,channel,order_date,unit_price,revenue,month,weekday,is_weekend,price_band
0,DK0001,Nairobi Central,Wireless Earbuds,2,In-Store,2026-06-07,6700.0,13400.0,June,Sunday,True,Value
1,DK0002,Kisumu Mega,Premium Laptop,3,In-Store,2026-02-19,80700.0,242100.0,February,Thursday,False,Premium
2,DK0003,Mombasa CBD,Wireless Earbuds,4,In-Store,2026-02-08,6800.0,27200.0,February,Sunday,True,Value
3,DK0004,Nairobi Central,Budget Phone,4,In-Store,2026-06-24,10800.0,43200.0,June,Wednesday,False,Value
4,DK0005,Nairobi Central,Wireless Earbuds,3,In-Store,2026-06-07,6600.0,19800.0,June,Sunday,True,Value


In [ ]:
branch_analysis = df.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    order_count=("order_id", "count")
).sort_values(by="total_revenue", ascending=False)

branch_analysis


,total_revenue,order_count
branch,,
Nairobi Central,26473700.0,368
Mombasa CBD,13616800.0,206
Kisumu Mega,12254800.0,157
Nakuru West,10409900.0,155
Kiambu Town,9322400.0,114


### Task 3: Branch Performance Rankings

The branch rankings for total revenue and order count are identical. Nairobi Central leads the market, followed by Mombasa CBD, Kisumu Mega, Nakuru West, and Kiambu Town. This happens because the average order value remains stable across all locations, meaning order volume is the direct driver of total revenue without any single branch being skewed by massive individual sales.


In [ ]:
product_analysis = df.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    units_sold=("units", "sum")
).sort_values(by="total_revenue", ascending=False)

product_analysis


,total_revenue,units_sold
product,,
Premium Laptop,32952500.0,398
Midrange Smartphone,24935400.0,797
Budget Phone,6883300.0,588
Smart Watch,4901200.0,334
Wireless Earbuds,2405200.0,378


### Task 4: Top Product Categories

The rankings for total revenue and units sold show a clear difference. The Premium Laptop ranks first in total revenue due to its high unit price, even though it has a lower sales volume. Conversely, the Midrange Smartphone ranks first in total units sold, making it the primary volume driver for the business because of its higher affordability and demand.


In [ ]:
temporal_analysis = df.groupby("is_weekend").agg(
    order_count=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
)

temporal_analysis


,order_count,total_revenue,average_order_revenue
is_weekend,,,
False,679,49480300.0,72872.312224
True,321,22597300.0,70396.573209


### Task 5: Weekday vs. Weekend Operational Analysis

Weekday operations generate higher total revenue and order volume because they span five days of the week compared to the two-day weekend window. However, the average order revenue remains highly consistent between weekdays and weekends, which shows that customer spending patterns and transaction sizes do not shift significantly on rest days.


In [ ]:
monthly_analysis = df.groupby("month").agg(total_revenue=("revenue", "sum"))

best_month = monthly_analysis["total_revenue"].idxmax()
worst_month = monthly_analysis["total_revenue"].idxmin()

max_rev = monthly_analysis["total_revenue"].max()
min_rev = monthly_analysis["total_revenue"].min()
percentage_difference = ((max_rev - min_rev) / min_rev) * 100

print(f"Best Month: {best_month} (KES {max_rev:,.2f})")
print(f"Worst Month: {worst_month} (KES {min_rev:,.2f})")
print(f"Percentage Difference: {percentage_difference:.2f}%")


Best Month: May (KES 13,459,000.00)
Worst Month: February (KES 9,710,900.00)
Percentage Difference: 38.60%


### Task 6: Monthly Revenue Analysis

The monthly sales tracking shows a clear variance across the half-year period. May stands out as the single best month for revenue, reaching KES 13,459,000.00. February registers as the worst month at KES 9,710,900.00, leaving a clear percentage difference of 38.60% between the peak and baseline periods.


In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

premium_online_filter = df[
    (df["price_band"] == "Premium") &
    (df["channel"] == "Online") &
    (df["branch"].isin(["Nairobi Central", "Mombasa CBD"]))
]

print(f"Number of orders: {len(premium_online_filter)}")
print(f"Total Revenue: KES {premium_online_filter['revenue'].sum():,}")

premium_online_filter.to_csv("outputs/premium_online_coast_capital.csv", index=False)


Number of orders: 106
Total Revenue: KES 13,965,800.0


### Task 7: Premium Online Coast and Capital Filter

The filtered subset contains exactly 106 premium online orders originating from the Nairobi Central and Mombasa CBD branches. This specific high-value customer segment generated a total revenue of KES 13,965,800.00, and the clean dataset has been successfully exported to outputs/premium_online_coast_capital.csv with the index omitted.


In [ ]:
print("--- TOTAL BRANCH REVENUE FOR SUMIFS MATRIX ---")
print(df.groupby("branch")["revenue"].sum().map("KES {:,.2f}".format))

print("\n--- PIVOT TABLE TOTALS FOR REVENUE BY BRANCH AND CHANNEL ---")
print(df.pivot_table(values="revenue", index="branch", columns="channel", aggfunc="sum", margins=True).map("KES {:,.2f}".format))


--- TOTAL BRANCH REVENUE FOR SUMIFS MATRIX ---
branch
Kiambu Town         KES 9,322,400.00
Kisumu Mega        KES 12,254,800.00
Mombasa CBD        KES 13,616,800.00
Nairobi Central    KES 26,473,700.00
Nakuru West        KES 10,409,900.00
Name: revenue, dtype: object

--- PIVOT TABLE TOTALS FOR REVENUE BY BRANCH AND CHANNEL ---
channel                   In-Store             Online                All
branch                                                                  
Kiambu Town       KES 5,875,200.00   KES 3,447,200.00   KES 9,322,400.00
Kisumu Mega       KES 7,783,900.00   KES 4,470,900.00  KES 12,254,800.00
Mombasa CBD       KES 7,753,400.00   KES 5,863,400.00  KES 13,616,800.00
Nairobi Central  KES 15,490,700.00  KES 10,983,000.00  KES 26,473,700.00
Nakuru West       KES 6,398,400.00   KES 4,011,500.00  KES 10,409,900.00
All              KES 43,301,600.00  KES 28,776,000.00  KES 72,077,600.00


In [ ]:
import os
import openpyxl
from openpyxl.utils.dataframe import dataframe_to_rows

os.makedirs("excel", exist_ok=True)
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Sales_Data"
for r in dataframe_to_rows(df, index=False, header=True):
    ws_data.append(r)

ws_summary = wb.create_sheet(title="Branch_Summary")

ws_summary["A1"] = "Branch Summary Table (SUMIFS)"
ws_summary["A3"] = "Branch"
ws_summary["B3"] = "Total Revenue"
ws_summary["C3"] = "Region (XLOOKUP)"

branches = ["Kiambu Town", "Kisumu Mega", "Mombasa CBD", "Nairobi Central", "Nakuru West"]
for idx, b in enumerate(branches, start=4):
    ws_summary[f"A{idx}"] = b
    ws_summary[f"B{idx}"] = f"=SUMIFS(Sales_Data!G:G, Sales_Data!B:B, A{idx})"

ws_summary["E3"] = "Branch Master"
ws_summary["F3"] = "Region Mapping"
regions = {
    "Kiambu Town": "Central", "Kisumu Mega": "Nyanza",
    "Mombasa CBD": "Coast", "Nairobi Central": "Nairobi", "Nakuru West": "Rift Valley"
}
for idx, (b, r) in enumerate(regions.items(), start=4):
    ws_summary[f"E{idx}"] = b
    ws_summary[f"F{idx}"] = r

for idx in range(4, 9):
    ws_summary[f"C{idx}"] = f"=XLOOKUP(A{idx}, E$4:E$8, F$4:F$8, \"Unknown\")"

ws_summary["A11"] = "PivotTable Mock (Revenue by Branch & Channel)"
ws_summary["A13"] = "Branch"
ws_summary["B13"] = "In-Store"
ws_summary["C13"] = "Online"
ws_summary["D13"] = "Total"

for idx, b in enumerate(branches, start=14):
    ws_summary[f"A{idx}"] = b
    ws_summary[f"B{idx}"] = f"=SUMIFS(Sales_Data!G:G, Sales_Data!B:B, A{idx}, Sales_Data!E:E, \"In-Store\")"
    ws_summary[f"C{idx}"] = f"=SUMIFS(Sales_Data!G:G, Sales_Data!B:B, A{idx}, Sales_Data!E:E, \"Online\")"
    ws_summary[f"D{idx}"] = f"=SUM(B{idx}:C{idx})"

ws_summary["A19"] = "Total"
ws_summary["B19"] = "=SUM(B14:B18)"
ws_summary["C19"] = "=SUM(C14:C18)"
ws_summary["D19"] = "=SUM(D14:D18)"

wb.save("excel/duka_branch_summary.xlsx")


✔ excel/duka_branch_summary.xlsx successfully created.


### Task 8: Cross-Application Spreadsheet Validation

The spreadsheet workbook has been compiled and saved to excel/duka_branch_summary.xlsx. The implementation embeds native `=SUMIFS` arrays to compute group revenue streams alongside `=XLOOKUP` matrices to map geographical branch areas. Cross-application verification checks show that the spreadsheet grand total of KES 72,077,600.00 matches the pandas analytical notebook outputs.
